In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
# 2. Carregamento dos Dados
products_df = spark.read.csv("olist_products_dataset.csv", header=True, inferSchema=True)
translation_df = spark.read.csv("product_category_name_translation.csv", header=True, inferSchema=True)

print("Schema do DataFrame de Produtos:")
products_df.printSchema()

print("\nSchema do DataFrame de Tradução:")
translation_df.printSchema()

Schema do DataFrame de Produtos:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)


Schema do DataFrame de Tradução:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [4]:
# 3. Remoção de Colunas
columns_to_drop = ["product_name_lenght", "product_description_lenght", "product_photos_qty"]
products_df = products_df.drop(*columns_to_drop)

print("DataFrame de Produtos após remoção de colunas:")
products_df.printSchema()

DataFrame de Produtos após remoção de colunas:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [5]:
# 4. Tradução de Categorias
# Juntando os dataframes para obter os nomes em inglês
products_translated_df = products_df.join(
    translation_df,
    on="product_category_name",
    how="left"
).withColumnRenamed(
    "product_category_name_english", "category_name"
).drop(
    "product_category_name"
)

# Reordenando as colunas para colocar 'category_name' depois de 'product_id'
final_columns = [
    "product_id",
    "category_name",
] + [c for c in products_translated_df.columns if c not in ["product_id", "category_name"]]

products_reordered_df = products_translated_df.select(final_columns)

print("DataFrame após tradução e reordenação das colunas:")
products_reordered_df.show(5)

DataFrame após tradução e reordenação das colunas:
+--------------------+--------------+----------------+-----------------+-----------------+----------------+
|          product_id| category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+--------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|     perfumery|             225|               16|               10|              14|
|3aa071139cb16b67c...|           art|            1000|               30|               18|              20|
|96bd76ec8810374ed...|sports_leisure|             154|               18|                9|              15|
|cef67bcfe19066a93...|          baby|             371|               26|                4|              26|
|9dc1a7de274444849...|    housewares|             625|               20|               17|              13|
+--------------------+--------------+----------------+-----------------+-------------

In [6]:
# 5. Cálculo do Volume
# Multiplicando as dimensões para criar a coluna 'volume'
# e removendo as colunas originais
products_volume_df = products_reordered_df.withColumn(
    "volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
).drop(
    "product_length_cm", "product_height_cm", "product_width_cm"
)

print("DataFrame final com a coluna de volume:")
products_volume_df.printSchema()

DataFrame final com a coluna de volume:
root
 |-- product_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- volume_cm3: integer (nullable = true)



In [7]:
# 6. Exibição do Resultado Final
final_df = products_volume_df.select("product_id", "category_name", "product_weight_g", "volume_cm3")

print("Amostra do DataFrame final transformado:")
final_df.show()

Amostra do DataFrame final transformado:
+--------------------+--------------------+----------------+----------+
|          product_id|       category_name|product_weight_g|volume_cm3|
+--------------------+--------------------+----------------+----------+
|1e9e8ef04dbcff454...|           perfumery|             225|      2240|
|3aa071139cb16b67c...|                 art|            1000|     10800|
|96bd76ec8810374ed...|      sports_leisure|             154|      2430|
|cef67bcfe19066a93...|                baby|             371|      2704|
|9dc1a7de274444849...|          housewares|             625|      4420|
|41d3672d4792049fa...| musical_instruments|             200|      2090|
|732bd381ad09e530f...|          cool_stuff|           18350|     73920|
|2548af3e6e77a690c...|     furniture_decor|             900|     12800|
|37cc742be07708b53...|     home_appliances|             400|      5967|
|8c92109888e8cdf9d...|                toys|             600|      2040|
|14aa47b7fe5c25522...| 

In [8]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

products_final_df = final_df.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"products_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
products_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\products_final_20260331_224555.csv
